In [ ]:
import fitz  # PyMuPDF
from langchain_text_splitters import RecursiveCharacterTextSplitter

PDF_PATH = "data/microsoft_2026.pdf" 
COMPANY = "Microsoft"
YEAR = 2026

CHUNK_SIZE = 900
CHUNK_OVERLAP = 120

In [ ]:
def extract_pages(pdf_path):
    doc = fitz.open(pdf_path)
    pages = []
    for page_no, page in enumerate(doc, start=1):
        pages.append({"page": page_no, "text": page.get_text()})
    return pages

pages = extract_pages(PDF_PATH)
print("Total pages:", len(pages))
print(pages[5]["text"][:600])

In [ ]:
empty = [p["page"] for p in pages if len(p["text"].strip()) < 50]
print("Nearly empty pages:", empty)

In [ ]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
)

def make_chunks(pages, company, year):
    chunks = []
    for p in pages:
        for i, piece in enumerate(splitter.split_text(p["text"])):
            if len(piece.strip()) < 30:      # skip junk like lone page numbers
                continue
            chunks.append({
                "id": f"{company}-{year}-p{p['page']}-{i}",
                "text": piece,
                "metadata": {"company": company, "year": year, "page": p["page"]},
            })
    return chunks

chunks = make_chunks(pages, COMPANY, YEAR)
print("Total chunks:", len(chunks))
print(chunks[50])

In [ ]:
lengths = [len(c["text"]) for c in chunks]
print("min:", min(lengths), "max:", max(lengths), "avg:", sum(lengths) // len(lengths))

In [ ]:
for c in chunks[200:204]:
    print(c["metadata"], "\n", c["text"], "\n" + "-" * 60)

In [ ]:
from sentence_transformers import SentenceTransformer

embedder = SentenceTransformer("BAAI/bge-small-en-v1.5")

In [ ]:
texts = [c["text"] for c in chunks]

embeddings = embedder.encode(texts, batch_size=32, normalize_embeddings=True, show_progress_bar=True)
print(embeddings.shape)

In [ ]:
import numpy as np

q = embedder.encode(
    "Represent this sentence for searching relevant passages: How many employees does Microsoft have?",
    normalize_embeddings=True,
)

scores = embeddings @ q          # similarity of the question to every chunk
top = np.argsort(scores)[::-1][:3]

for i in top:
    print(round(float(scores[i]), 3), "| page", chunks[i]["metadata"]["page"], "|", chunks[i]["text"][:100])

In [11]:
import chromadb

client = chromadb.PersistentClient("chroma_db")
collection = client.get_or_create_collection("reports", metadata={"hnsw:space": "cosine"})

collection.upsert(
    ids=[c["id"] for c in chunks],
    documents=texts,
    embeddings=embeddings.tolist(),
    metadatas=[c["metadata"] for c in chunks],
)

print("Stored:", collection.count())

Stored: 580


In [12]:
QUERY_PREFIX = "Represent this sentence for searching relevant passages: "

def retrieve(question, k=6):
    q_emb = embedder.encode(QUERY_PREFIX + question, normalize_embeddings=True).tolist()
    res = collection.query(query_embeddings=[q_emb], n_results=k)
    return [
        {"text": doc, "distance": dist, **meta}
        for doc, dist, meta in zip(res["documents"][0], res["distances"][0], res["metadatas"][0])
    ]

for r in retrieve("What was total revenue for fiscal year 2026?"):
    print(f"p.{r['page']} | dist {r['distance']:.3f} | {r['text'][:100]}")

p.35 | dist 0.253 | businesses; licensing and supporting an array of software products; delivering relevant online adver
p.80 | dist 0.260 | 2026, 2025, and 2024 by $21.6 billion, $21.2 billion, and $19.6 billion, respectively. 
As of June 3
p.52 | dist 0.261 | PART II 
Item 8 
51 
 
ITEM 8. FINANCIAL STATEMENTS AND SUPPLEMENTARY DATA  
INCOME STATEMENTS 
 
(I
p.90 | dist 0.272 | 23,481 
25,238
24,892 
Operating expenses 
16,185 
15,245
13,987 
Operating income 
$ 
14,386 $ 
14,
p.38 | dist 0.276 | financial results. 
Fiscal Year 2026 Compared with Fiscal Year 2025 
Revenue increased $50.1 billion
p.76 | dist 0.279 | (In millions) 
 
 
 
 
 
 
 
Year Ended June 30, 
 
2026  
 
2025  
 
2024  
U.S. 
 $ 103,591 
 $ 69


In [ ]:
import time
from google.genai import errors

MODELS = ["gemini-2.5-flash", "gemini-2.5-flash-lite","gemini-2.5-flash","gemini-3.5-flash"]   # tried in order

def generate(prompt, temperature=0):
    last_error = None
    for model in MODELS:
        for attempt in range(3):
            try:
                r = llm.models.generate_content(
                    model=model,
                    contents=prompt,
                    config=types.GenerateContentConfig(temperature=temperature),
                )
                return r.text
            except (errors.ServerError, errors.ClientError) as e:
                last_error = e
                code = getattr(e, "code", None)
                if code in (429, 503):          # busy or rate-limited: wait, retry
                    time.sleep(5 * (attempt + 1))
                    continue
                break                            # other error (e.g. 404): try next model
    raise last_error

print(generate("Say hello in one word."))

ClientError: 404 NOT_FOUND. {'error': {'code': 404, 'message': 'This model models/gemini-2.5-flash is no longer available to new users. Please update your code to use models/gemini-3.8-flash for the latest features and improvements. We recommend you to use the Interactions API (https://ai.google.dev/gemini-api/docs/get-started).', 'status': 'NOT_FOUND'}}

In [16]:
for m in llm.models.list():
    if "generateContent" in (m.supported_actions or []):
        print(m.name)

models/gemini-2.5-flash
models/gemini-2.5-pro
models/gemini-2.5-flash-preview-tts
models/gemini-2.5-pro-preview-tts
models/gemma-4-26b-a4b-it
models/gemma-4-31b-it
models/gemini-flash-latest
models/gemini-flash-lite-latest
models/gemini-pro-latest
models/gemini-2.5-flash-lite
models/gemini-2.5-flash-image
models/gemini-3-flash-preview
models/gemini-3.1-pro-preview
models/gemini-3.1-pro-preview-customtools
models/gemini-3.1-flash-lite-preview
models/gemini-3.1-flash-lite
models/gemini-3-pro-image-preview
models/gemini-3-pro-image
models/nano-banana-pro-preview
models/gemini-3.1-flash-image-preview
models/gemini-3.1-flash-image
models/gemini-3.1-flash-lite-image
models/gemini-nano-banana-2.1
models/gemini-3.5-flash
models/gemini-3.5-flash-lite
models/gemini-omni-flash-preview
models/gemini-omni-1.1-flash
models/gemini-3.5-transcribe
models/gemini-3.6-flash
models/gemini-3.7-flash
models/gemini-3.8-flash
models/gemini-4-argon
models/lyria-3-clip-preview
models/lyria-3-pro-preview
models/l